<a href="https://colab.research.google.com/github/nivethithanm/mini-claw/blob/main/CLAW_04_skill_system.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# CLAW-04 — Skill System: Extensible Agent Capabilities

**Goal:** Build OpenClaw's killer feature — a dynamic skill system where capabilities
can be added, removed, or even self-written at runtime.

By the end of this notebook you'll have:
- A `Skill` abstraction: a named bundle of tools + a description
- A `SkillLoader` that discovers skills from the filesystem
- A `SkillRegistry` that manages the active skill set
- 3 built-in skills: `system_info`, `file_ops`, `web_research`
- Hot-reload: add a new skill file while the agent is running

> **First principles mindset:**  
> A skill is just a Python file that defines tools.  
> The agent's capabilities are the union of all loaded skill files.  
> This is how OpenClaw lets users add skills by dropping files into a folder.


## 0. Setup

In [1]:
import os, json, importlib, importlib.util, subprocess, platform
from pathlib import Path
from dataclasses import dataclass, field
from typing import Callable

SKILLS_DIR = Path("./claw_skills")
SKILLS_DIR.mkdir(exist_ok=True)
print(f"Skills dir: {SKILLS_DIR.resolve()}")


Skills dir: /content/claw_skills


## 1. The Skill Abstraction

A skill groups related tools together under a name.

```
Skill: "file_ops"
  - Tools: read_file, write_file, list_dir, delete_file
  - Description: "Read and write files on the local filesystem."
```


In [2]:
@dataclass
class Skill:
    """
    A named bundle of related tools.

    Skills are the unit of extension in our agent.
    Users can add skills by dropping Python files into ~/.claw/skills/
    """
    name: str
    description: str
    tools: list  # list of Tool objects (from NB-02)
    version: str = "1.0.0"
    author: str = "built-in"

    def tool_names(self) -> list[str]:
        return [t.name for t in self.tools]

    def __repr__(self):
        return f"Skill('{self.name}', tools={self.tool_names()})"


## 2. Built-in Skill: system_info

Tools for querying the local machine. The agent can tell you what OS you're on,
check disk space, list running processes.


In [5]:
import platform, shutil, subprocess, os

def make_system_info_skill():
    """
    Build the system_info skill.

    Exercise: implement these tool functions:
    - _get_os_info() -> str  (OS name, version, architecture)
    - _get_disk_usage() -> str  (total/used/free for current directory)
    - _run_command(command: str) -> str  (run a whitelisted shell command)

    Whitelist for _run_command: ["ls", "pwd", "date", "whoami", "uname", "df", "uptime"]
    """

    WHITELIST = {"ls", "pwd", "date", "whoami", "uname", "df", "uptime"}

    def _get_os_info() -> str:
        # TODO: use platform module
        os_name = platform.system()
        os_version = platform.release()
        os_arch = platform.machine()
        python_version = platform.python_version()
        return f"{os_name} {os_version} ({os_arch}), Python {python_version}"

    def _get_disk_usage(path: str = ".") -> str:
        # TODO: use shutil.disk_usage()
        disk_usage = shutil.disk_usage(path)
        gb = 1024**3
        return (f"Total: {disk_usage.total/gb:.1f}GB | "
                f"Used: {disk_usage.used/gb:.1f}GB | "
                f"Free: {disk_usage.free/gb:.1f}GB")

    def _run_command(command: str) -> str:
        """Run a whitelisted shell command safely."""
        cmd_name = command.strip().split()[0]
        if cmd_name not in WHITELIST:
            return f"Error: '{cmd_name}' not in whitelist {WHITELIST}"
        # TODO: use subprocess.run with timeout=5, capture_output=True
        try:
            result = subprocess.run(
                command, shell=True, capture_output=True, text=True, timeout=5
            )
            return result.stdout or result.stderr or "(no output)"
        except subprocess.TimeoutExpired:
            return "Error: command timed out"
        except Exception as e:
            return f"Error: {e}"

    # Build Tool objects (simplified inline version)
    class T:
        def __init__(self, name, desc, params, req, fn):
            self.name, self.description, self.parameters = name, desc, params
            self.required, self.fn = req, fn
        def to_openai_spec(self):
            return {"type": "function", "function": {
                "name": self.name, "description": self.description,
                "parameters": {"type": "object", "properties": self.parameters, "required": self.required}
            }}
        def execute(self, **kw):
            try: return str(self.fn(**kw))
            except Exception as e: return f"Error: {e}"

    tools = [
        T("get_os_info", "Get OS name, version, and architecture.", {}, [], _get_os_info),
        T("get_disk_usage", "Get disk usage for a path.", {"path": {"type": "string", "default": "."}}, [], _get_disk_usage),
        T("run_command", "Run a whitelisted shell command.", {"command": {"type": "string"}}, ["command"], _run_command),
    ]

    return Skill(
        name="system_info",
        description="Query the local machine: OS info, disk usage, shell commands.",
        tools=tools,
    )


In [6]:
skill = make_system_info_skill()
print(skill)
for t in skill.tools:
    print(f"  Testing {t.name}:", t.execute())
print(t.execute(command="ls -la"))


Skill('system_info', tools=['get_os_info', 'get_disk_usage', 'run_command'])
  Testing get_os_info: Linux 6.6.122+ (x86_64), Python 3.12.13
  Testing get_disk_usage: Total: 107.7GB | Used: 20.3GB | Free: 87.4GB
  Testing run_command: Error: make_system_info_skill.<locals>._run_command() missing 1 required positional argument: 'command'
total 20
drwxr-xr-x 1 root root 4096 Jun  7 10:09 .
drwxr-xr-x 1 root root 4096 Jun  7 09:56 ..
drwxr-xr-x 2 root root 4096 Jun  7 10:09 claw_skills
drwxr-xr-x 4 root root 4096 Jun  4 13:32 .config
drwxr-xr-x 1 root root 4096 Jun  4 13:32 sample_data



## 3. Built-in Skill: file_ops

In [7]:
def make_file_ops_skill():
    """
    Exercise: build a file_ops skill with these tools:
    - read_file(path): read a file, return first 3000 chars
    - write_file(path, content): write/overwrite a file
    - list_dir(path="."): list directory contents
    - append_file(path, content): append to a file

    Safety: restrict paths to current working directory and subdirectories.
    """
    def _safe_path(path: str) -> Path:
        p = Path(path).resolve()
        cwd = Path(".").resolve()
        if not str(p).startswith(str(cwd)):
            raise ValueError(f"Path outside working directory: {path}")
        return p

    class T:
        def __init__(self, name, desc, params, req, fn):
            self.name, self.description, self.parameters = name, desc, params
            self.required, self.fn = req, fn
        def to_openai_spec(self):
            return {"type": "function", "function": {
                "name": self.name, "description": self.description,
                "parameters": {"type": "object", "properties": self.parameters, "required": self.required}
            }}
        def execute(self, **kw):
            try: return str(self.fn(**kw))
            except Exception as e: return f"Error: {e}"

    def _read_file(path: str, max_chars: int = 3000) -> str:
        p = _safe_path(path)
        return p.read_text()[:max_chars]

    def _write_file(path: str, content: str) -> str:
        p = _safe_path(path)
        p.parent.mkdir(parents=True, exist_ok=True)
        p.write_text(content)
        return f"Written {len(content)} chars to {path}"

    def _list_dir(path: str = ".") -> str:
        p = _safe_path(path)
        items = sorted(p.iterdir(), key=lambda x: (x.is_file(), x.name))
        lines = []
        for item in items:
            prefix = "📄" if item.is_file() else "📁"
            lines.append(f"{prefix} {item.name}")
        return "\n".join(lines) or "(empty)"

    def _append_file(path: str, content: str) -> str:
        p = _safe_path(path)
        with open(p, "a") as f:
            f.write(content)
        return f"Appended {len(content)} chars to {path}"

    tools = [
        T("read_file", "Read file contents.", {"path": {"type": "string"}, "max_chars": {"type": "integer"}}, ["path"], _read_file),
        T("write_file", "Write content to file.", {"path": {"type": "string"}, "content": {"type": "string"}}, ["path", "content"], _write_file),
        T("list_dir", "List directory contents.", {"path": {"type": "string"}}, [], _list_dir),
        T("append_file", "Append to file.", {"path": {"type": "string"}, "content": {"type": "string"}}, ["path", "content"], _append_file),
    ]
    return Skill(name="file_ops", description="Read, write, and list files on the local filesystem.", tools=tools)


In [14]:
file_skill = make_file_ops_skill()
print(file_skill)
print(file_skill.tools[2].execute(path="./claw_skills"))  # list


Skill('file_ops', tools=['read_file', 'write_file', 'list_dir', 'append_file'])
(empty)


## 4. SkillRegistry

In [15]:
class SkillRegistry:
    """
    Manages the active set of skills and exposes all their tools.

    This is the object passed to the agent loop instead of a flat ToolRegistry.
    """
    def __init__(self):
        self._skills: dict[str, Skill] = {}

    def load(self, skill: Skill):
        """Add a skill."""
        self._skills[skill.name] = skill
        print(f"  ✓ Loaded skill: {skill.name} ({len(skill.tools)} tools)")

    def unload(self, skill_name: str):
        """Remove a skill. Exercise: implement this."""
        raise NotImplementedError

    def all_tools(self) -> list:
        """Return all tools from all loaded skills."""
        tools = []
        for skill in self._skills.values():
            tools.extend(skill.tools)
        return tools

    def to_openai_specs(self) -> list[dict]:
        """All tool specs for the OpenAI API."""
        return [t.to_openai_spec() for t in self.all_tools()]

    def dispatch(self, name: str, arguments: str | dict) -> str:
        """Find and execute a tool by name across all skills."""
        for tool in self.all_tools():
            if tool.name == name:
                kwargs = json.loads(arguments) if isinstance(arguments, str) else arguments
                return tool.execute(**kwargs)
        return f"Error: unknown tool '{name}'"

    def status(self):
        print(f"Skills loaded: {len(self._skills)}")
        for name, skill in self._skills.items():
            print(f"  [{name}] {skill.description}")
            for t in skill.tools:
                print(f"    - {t.name}")


# Test
reg = SkillRegistry()
reg.load(make_system_info_skill())
reg.load(make_file_ops_skill())
reg.status()


  ✓ Loaded skill: system_info (3 tools)
  ✓ Loaded skill: file_ops (4 tools)
Skills loaded: 2
  [system_info] Query the local machine: OS info, disk usage, shell commands.
    - get_os_info
    - get_disk_usage
    - run_command
  [file_ops] Read, write, and list files on the local filesystem.
    - read_file
    - write_file
    - list_dir
    - append_file


## 5. Skill Loader — Hot-Reload from Filesystem

OpenClaw lets users add skills by dropping a Python file into `~/.claw/skills/`.
Each file just needs to expose a `build_skill()` function.


In [17]:
EXAMPLE_SKILL = """
# weather skill
import urllib.request, json

def build_skill():
    def _get_weather(city):
        url = f'https://wttr.in/{city}?format=3'
        try:
            with urllib.request.urlopen(url, timeout=5) as r:
                return r.read().decode()
        except Exception as e:
            return f'Error: {e}'

    class T:
        def __init__(self,name,desc,params,req,fn):
            self.name,self.description,self.parameters=name,desc,params
            self.required,self.fn=req,fn
        def to_openai_spec(self):
            return {'type':'function','function':{
                'name':self.name,'description':self.description,
                'parameters':{
                    'type':'object','properties':self.parameters,'required':self.required}}}
        def execute(self,**kw): return str(self.fn(**kw))

    from __main__ import Skill
    return Skill(name='weather',description='Get weather.',
        tools=[T('get_weather','Get weather.',{'city':
            {'type':'string'}},['city'],_get_weather)])
"""

skill_file = SKILLS_DIR / "weather_skill.py"
skill_file.write_text(EXAMPLE_SKILL)
print(f"Wrote skill file: {skill_file}")

Wrote skill file: claw_skills/weather_skill.py


In [18]:
class SkillLoader:
    """
    Discovers and loads skill files from a directory.

    Each .py file in the skills directory should export build_skill() -> Skill.
    """

    def __init__(self, skills_dir: Path = SKILLS_DIR):
        self.skills_dir = skills_dir
        self._loaded_mtimes: dict[str, float] = {}

    def load_all(self, registry: SkillRegistry):
        """Scan skills_dir and load all valid skill files into registry."""
        for path in self.skills_dir.glob("*.py"):
            self._load_file(path, registry)

    def _load_file(self, path: Path, registry: SkillRegistry):
        """
        Dynamically import a skill file and call its build_skill().

        Exercise: use importlib.util to load the module from path,
        call build_skill(), and load into registry.
        Handle ImportError and AttributeError gracefully.
        """
        try:
            spec = importlib.util.spec_from_file_location(path.stem, path)
            module = importlib.util.module_from_spec(spec)
            spec.loader.exec_module(module)
            if hasattr(module, "build_skill"):
                skill = module.build_skill()
                registry.load(skill)
                self._loaded_mtimes[str(path)] = path.stat().st_mtime
            else:
                print(f"  ⚠ {path.name}: no build_skill() found, skipping")
        except Exception as e:
            print(f"  ✗ Failed to load {path.name}: {e}")

    def reload_changed(self, registry: SkillRegistry):
        """
        Hot-reload: check for modified or new skill files and reload them.
        Exercise: compare file mtime to self._loaded_mtimes, reload if changed.
        """
        for path in self.skills_dir.glob("*.py"):
            mtime = path.stat().st_mtime
            if self._loaded_mtimes.get(str(path)) != mtime:
                print(f"  🔄 Hot-reloading: {path.name}")
                self._load_file(path, registry)


In [19]:
# Test hot-load
reg2 = SkillRegistry()
loader = SkillLoader()
loader.load_all(reg2)
print()
reg2.status()

  ✓ Loaded skill: weather (1 tools)

Skills loaded: 1
  [weather] Get weather.
    - get_weather


## 6. Exercises

**E1.** Write a `note_taking` skill with tools: `create_note(title, content)`, `list_notes()`, `read_note(title)`, `search_notes(query)`. Notes are stored as Markdown files.

**E2.** Write a `calendar` skill that reads/writes events to a local JSON file. Tools: `add_event`, `list_events(date)`, `delete_event`.

**E3.** Implement **skill sandboxing**: each skill runs in a separate thread with a 10-second timeout. If a tool call exceeds the timeout, return an error message rather than hanging.

**E4 (hard):** Let the agent write its own skills. Implement a `create_skill(name, description, tools_spec)` meta-tool that:
1. Takes a description of what the skill should do
2. Generates the Python skill file using the LLM
3. Saves it to the skills directory
4. Hot-reloads it
This is exactly how OpenClaw's self-building capability works.

---

## ✅ Checkpoint

- `Skill`: named bundle of tools
- `SkillRegistry`: manages skills, dispatches tool calls  
- `SkillLoader`: discovers skills from filesystem, supports hot-reload
- 2 built-in skills: `system_info`, `file_ops`

**Next:** CLAW-05 — Shell & Browser. Full computer control.
